In [0]:
%sql
-- =============================================================
-- Projet Retail Azure - Couche GOLD
-- Source : retail.silver.*
-- Cible  : modèle en étoile retail.gold.* + vues KPI pour Power BI
-- Limite connue : pas de prix au moment de la vente dans la source,
--                 le montant est calculé avec le prix catalogue actuel.
-- =============================================================

USE CATALOG retail;
USE SCHEMA gold;

-- -------------------------------------------------------------
-- 1. Dimensions
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE retail.gold.dim_customer
COMMENT 'Dimension client'
AS
SELECT customer_id,
       first_name,
       last_name,
       CONCAT(first_name, ' ', last_name) AS customer_name,
       email,
       city,
       registration_date
FROM retail.silver.customers;

CREATE OR REPLACE TABLE retail.gold.dim_product
COMMENT 'Dimension produit'
AS
SELECT product_id, product_name, category, price
FROM retail.silver.products;

CREATE OR REPLACE TABLE retail.gold.dim_store
COMMENT 'Dimension magasin'
AS
SELECT store_id, store_name, city
FROM retail.silver.stores;

CREATE OR REPLACE TABLE retail.gold.dim_date
COMMENT 'Dimension calendrier couvrant la période des ventes'
AS
WITH bounds AS (
  SELECT MIN(transaction_date) AS d_min, MAX(transaction_date) AS d_max
  FROM retail.silver.transactions
)
SELECT
  d                                    AS date,
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  YEAR(d)                              AS year,
  QUARTER(d)                           AS quarter,
  MONTH(d)                             AS month,
  date_format(d, 'MMMM')               AS month_name,
  date_format(d, 'yyyy-MM')            AS year_month,
  DAYOFWEEK(d)                         AS day_of_week,
  date_format(d, 'EEEE')               AS day_name,
  DAYOFWEEK(d) IN (1, 7)               AS is_weekend
FROM bounds
LATERAL VIEW explode(sequence(d_min, d_max, INTERVAL 1 DAY)) AS d;

-- -------------------------------------------------------------
-- 2. Table de faits
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE retail.gold.fact_sales
COMMENT 'Faits de vente - une ligne par transaction'
AS
SELECT
  t.transaction_id,
  CAST(date_format(t.transaction_date, 'yyyyMMdd') AS INT) AS date_key,
  t.transaction_date,
  t.customer_id,
  t.product_id,
  t.store_id,
  t.quantity,
  p.price                              AS unit_price,
  CAST(t.quantity * p.price AS DECIMAL(12,2)) AS sales_amount
FROM retail.silver.transactions t
JOIN retail.silver.products p ON t.product_id = p.product_id;

-- -------------------------------------------------------------
-- 3. Vues KPI (besoins métier du projet)
-- -------------------------------------------------------------

-- Ventes totales par magasin et catégorie (sert aussi à la heatmap)
CREATE OR REPLACE VIEW retail.gold.vw_sales_by_store_category AS
SELECT s.store_name,
       s.city,
       p.category,
       SUM(f.quantity)      AS total_quantity,
       SUM(f.sales_amount)  AS total_sales
FROM retail.gold.fact_sales f
JOIN retail.gold.dim_store   s ON f.store_id   = s.store_id
JOIN retail.gold.dim_product p ON f.product_id = p.product_id
GROUP BY s.store_name, s.city, p.category;

-- Tendance journalière des ventes par produit
CREATE OR REPLACE VIEW retail.gold.vw_daily_sales_by_product AS
SELECT f.transaction_date,
       p.product_name,
       p.category,
       SUM(f.quantity)      AS total_quantity,
       SUM(f.sales_amount)  AS total_sales
FROM retail.gold.fact_sales f
JOIN retail.gold.dim_product p ON f.product_id = p.product_id
GROUP BY f.transaction_date, p.product_name, p.category;

-- Panier moyen par magasin
CREATE OR REPLACE VIEW retail.gold.vw_avg_order_value_by_store AS
SELECT s.store_name,
       s.city,
       COUNT(DISTINCT f.transaction_id)                          AS nb_orders,
       SUM(f.sales_amount)                                       AS total_sales,
       ROUND(SUM(f.sales_amount) / COUNT(DISTINCT f.transaction_id), 2) AS avg_order_value
FROM retail.gold.fact_sales f
JOIN retail.gold.dim_store s ON f.store_id = s.store_id
GROUP BY s.store_name, s.city;

-- -------------------------------------------------------------
-- 4. Contrôles
--    Attendu : fact_sales 30 lignes, sales_amount non nul, dim_date > 0
-- -------------------------------------------------------------
SELECT
  (SELECT COUNT(*) FROM retail.gold.fact_sales)                       AS fact_rows,
  (SELECT SUM(sales_amount) FROM retail.gold.fact_sales)              AS total_sales,
  (SELECT COUNT(*) FROM retail.gold.fact_sales WHERE sales_amount IS NULL) AS null_amounts,
  (SELECT COUNT(*) FROM retail.gold.dim_date)                         AS calendar_days;

SELECT * FROM retail.gold.vw_avg_order_value_by_store ORDER BY avg_order_value DESC;